# PDDL Generation Prompt Prototyping

## Environment Setup

In [ ]:
import os
import re
from random import Random
from copy import deepcopy
from ipywidgets import interact
from jinja2 import Environment, FileSystemLoader

In [ ]:
DATA_PATH = '../../resources/data/CVE-PDDL'
AP_PATTERN = re.compile(r'AP\d+')
NL_DESCRIPTION_FILE = 'description.txt'
DOMAIN_FILE = 'domain.pddl'
PROBLEM_FILE = 'problem.pddl'

TEMPLATES_PATH = '../../resources/prompt/generation'
COMPLETION_TEMPLATE_FILE = 'completion.md.jinja'
CHAT_TEMPLATE_FILE = 'chat.jsonl.jinja'

ID = 'id'
DESCRIPTION = 'description'
ATTACK_PATHS = 'attack_paths'
DOMAIN = 'domain'
GOAL = 'goal'
PROBLEM = 'problem'

SEED = 42

In [ ]:
def load_from_text_file(path: str) -> str:
    with open(path) as f:
        content = f.read()

    return content

In [ ]:
env = Environment(loader=FileSystemLoader(TEMPLATES_PATH))

## Data

In [ ]:
data = [
    {
        ID: cve,
        DESCRIPTION: load_from_text_file(os.path.join(DATA_PATH, cve, NL_DESCRIPTION_FILE)),
        ATTACK_PATHS: [
            {
                DOMAIN: load_from_text_file(os.path.join(DATA_PATH, cve, ap, DOMAIN_FILE)),
                GOAL: '[GOAL]',  # TODO read goal from problem file
                PROBLEM: load_from_text_file(os.path.join(DATA_PATH, cve, ap, PROBLEM_FILE))
            }
            for ap in os.listdir(os.path.join(DATA_PATH, cve)) if AP_PATTERN.match(ap)
        ]
    } 
    for cve in os.listdir(DATA_PATH)
]

## Prompt

### Completion

In [ ]:
n = 0  # Nuber of shots
c = DOMAIN  # One of domain, goal, problem

def render_completion_template(n: int, c: str):
    rng = Random(SEED)
    target, *examples = rng.choices(deepcopy(data), k=n + 1)

    cves = examples + [target]
    for idx, cve in enumerate(cves):
        aps = cve.pop(ATTACK_PATHS)
        ap = rng.choice(aps)
        if idx == len(cves) - 1:
            if c == DOMAIN or c == GOAL or c == PROBLEM:
                ap.pop(PROBLEM)
            if c == DOMAIN or c == GOAL:
                ap.pop(GOAL)
            if c == DOMAIN:
                ap.pop(DOMAIN)
        cve |= ap

    template = env.get_template(COMPLETION_TEMPLATE_FILE)
    rendered_content = template.render(cves=cves)
    print(rendered_content)

interact(
    render_completion_template,
    x=(0, 3, 1),  # Slider
    choice=[DOMAIN, GOAL, PROBLEM]  # Dropdown
)

### Chat

In [ ]:
...  # TBD